In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Read Streaming Data**

In [0]:

df_batch = (spark.read
  .format("json")
  .option("multiLine", True)
  .load("/Volumes/databricks_streaming/stream/streaming/jsonsource")
)


df_batch.printSchema()


print(df_batch.schema.simpleString())

root
 |-- customer: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- postal_code: string (nullable = true)
 |    |-- customer_id: long (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |-- metadata: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- key: string (nullable = true)
 |    |    |-- value: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- method: string (nullable = true)
 |    |-- transaction_id: string (nullable = true)


In [0]:
my_schema = """
    customer STRUCT<
        address STRUCT<
            city STRING,
            country STRING,
            postal_code STRING
        >,
        customer_id BIGINT,
        email STRING,
        name STRING
    >,
    items ARRAY<STRUCT<
        item_id STRING,
        price DOUBLE,
        product_name STRING,
        quantity BIGINT
    >>,
    metadata ARRAY<STRUCT<
        key STRING,
        value STRING
    >>,
    order_id STRING,
    payment STRUCT<
        method STRING,
        transaction_id STRING
    >,
    timestamp STRING
"""

In [0]:
df = (spark.read
  .format("json")
  .option("multiLine", True)
  .schema(my_schema)
  .load("/Volumes/databricks_streaming/stream/streaming/jsonsource")
)

display(df)

customer,items,metadata,order_id,payment,timestamp
"List(List(Toronto, Canada, M5H 2N2), 501, john@example.com, John Doe)","List(List(I100, 25.99, Wireless Mouse, 2), List(I101, 15.49, USB-C Adapter, 1))","List(List(campaign, back_to_school), List(channel, email))",ORD1001,"List(Credit Card, TXN7890)",2025-06-01T10:15:00Z


In [0]:
display(df.select("customer.*"))

address,customer_id,email,name
"List(Toronto, Canada, M5H 2N2)",501,john@example.com,John Doe


In [0]:
display(df.select("items"))

items
"List(List(I100, 25.99, Wireless Mouse, 2), List(I101, 15.49, USB-C Adapter, 1))"


In [0]:
# Transformations
df = df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.country","customer.address.postal_code","payment","metadata")
display(df)

items,order_id,timestamp,customer_id,name,email,city,country,postal_code,payment,metadata
"List(List(I100, 25.99, Wireless Mouse, 2), List(I101, 15.49, USB-C Adapter, 1))",ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,"List(Credit Card, TXN7890)","List(List(campaign, back_to_school), List(channel, email))"


In [0]:
df = df.withColumn("items",explode_outer("items"))
display(df)

items,order_id,timestamp,customer_id,name,email,city,country,postal_code,payment,metadata
"List(I100, 25.99, Wireless Mouse, 2)",ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,"List(Credit Card, TXN7890)","List(List(campaign, back_to_school), List(channel, email))"
"List(I101, 15.49, USB-C Adapter, 1)",ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,"List(Credit Card, TXN7890)","List(List(campaign, back_to_school), List(channel, email))"


In [0]:
df = df.select("items.item_id","items.price","items.product_name","items.quantity","order_id","timestamp","customer_id","name","email","city","country","postal_code","payment.method","payment.transaction_id","metadata")

In [0]:
display(df)

item_id,price,product_name,quantity,order_id,timestamp,customer_id,name,email,city,country,postal_code,method,transaction_id,metadata
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(List(campaign, back_to_school), List(channel, email))"
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(List(campaign, back_to_school), List(channel, email))"


In [0]:
df = df.withColumn("metadata",explode_outer("metadata"))
display(df)

item_id,price,product_name,quantity,order_id,timestamp,customer_id,name,email,city,country,postal_code,method,transaction_id,metadata
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(campaign, back_to_school)"
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(channel, email)"
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(campaign, back_to_school)"
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(channel, email)"


In [0]:
df = df.select("*","metadata.key","metadata.value").drop("metadata")
display(df)

item_id,price,product_name,quantity,order_id,timestamp,customer_id,name,email,city,country,postal_code,method,transaction_id,key,value
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email


In [0]:
df = spark.readStream.format("json")\
          .option("multiLine",True)\
          .schema(my_schema)\
          .load("/Volumes/databricks_streaming/stream/streaming/jsonsource")


# Transformations
df = df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.country","customer.address.postal_code","payment","metadata")

df = df.withColumn("items",explode_outer("items"))

df = df.select("items.item_id","items.price","items.product_name","items.quantity","order_id","timestamp","customer_id","name","email","city","country","postal_code","payment.method","payment.transaction_id","metadata")

df = df.withColumn("metadata",explode_outer("metadata"))
df = df.select("*","metadata.key","metadata.value").drop("metadata")


In [0]:
df.writeStream.format("delta")\
        .outputMode("append")\
        .trigger(once=True)\
        .option("path","/Volumes/databricks_streaming/stream/streaming/jsonsink/Data")\
        .option("checkpointLocation","/Volumes/databricks_streaming/stream/streaming/jsonsink/checkpoint")\
        .start()


In [0]:
%sql
SELECT * FROM delta.`/Volumes/databricks_streaming/stream/streaming/jsonsink/Data`

item_id,price,product_name,quantity,order_id,timestamp,customer_id,name,email,city,country,postal_code,method,transaction_id,key,value
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email
I102,45.0,Bluetooth Keyboard,1,ORD1002,2025-06-01T10:30:00Z,502,Alice Smith,alice@example.com,Vancouver,Canada,V5K 0A1,PayPal,TXN7891,campaign,cyber_monday
I102,45.0,Bluetooth Keyboard,1,ORD1002,2025-06-01T10:30:00Z,502,Alice Smith,alice@example.com,Vancouver,Canada,V5K 0A1,PayPal,TXN7891,channel,affiliate


### **Archiving**

In [0]:
dbutils.fs.mkdirs("/Volumes/databricks_streaming/stream/streaming/jsonsinknew")


True

In [0]:
dbutils.fs.mkdirs("/Volumes/databricks_streaming/stream/streaming/jsonsourcearchive")

True

In [0]:
df = spark.readStream.format("json")\
        .option("multiLine",True)\
        .schema(my_schema)\
        .option("cleanSource","archive")\
        .option("sourceArchiveDir","/Volumes/databricks_streaming/stream/streaming/jsonsourcearchive")\
        .load("/Volumes/databricks_streaming/stream/streaming/jsonsourcenew")


# Transformations
df = df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.country","customer.address.postal_code","payment","metadata")

df = df.withColumn("items",explode_outer("items"))

df = df.select("items.item_id","items.price","items.product_name","items.quantity","order_id","timestamp","customer_id","name","email","city","country","postal_code","payment.method","payment.transaction_id","metadata")

df = df.withColumn("metadata",explode_outer("metadata"))
df = df.select("*","metadata.key","metadata.value").drop("metadata")


In [0]:
df.writeStream.format("delta")\
        .outputMode("append")\
        .trigger(once=True)\
        .option("path","/Volumes/databricks_streaming/stream/streaming/jsonsinknew/Data")\
        .option("checkpointLocation","/Volumes/databricks_streaming/stream/streaming/jsonsinknew/checkpoint")\
        .start()



### READ JSON DATA (BATCH)

In [0]:
# df = spark.read.format("json")\
#           .option("inferShcema",True)\
#           .option("multiLine",True)\
#           .load("/Volumes/databricks_streaming/stream/streaming/jsonsource")

# display(df)

In [0]:
# df = df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.country","customer.address.postal_code","payment","metadata")

# df = df.withColumn("items",explode_outer("items"))

# display(df)

items,order_id,timestamp,customer_id,name,email,city,country,postal_code,payment,metadata
"List(I100, 25.99, Wireless Mouse, 2)",ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,"List(Credit Card, TXN7890)","List(List(campaign, back_to_school), List(channel, email))"
"List(I101, 15.49, USB-C Adapter, 1)",ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,"List(Credit Card, TXN7890)","List(List(campaign, back_to_school), List(channel, email))"


In [0]:
# df = df.select("items.item_id","items.price","items.product_name","items.quantity","order_id","timestamp","customer_id","name","email","city","country","postal_code","payment.method","payment.transaction_id","metadata")
# display(df)

item_id,price,product_name,quantity,order_id,timestamp,customer_id,name,email,city,country,postal_code,method,transaction_id,metadata
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(List(campaign, back_to_school), List(channel, email))"
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(List(campaign, back_to_school), List(channel, email))"


In [0]:
# df = df.withColumn("metadata",explode_outer("metadata"))
# df = df.select("*","metadata.key","metadata.value").drop("metadata")
# display(df)

item_id,price,product_name,quantity,order_id,timestamp,customer_id,name,email,city,country,postal_code,method,transaction_id,key,value
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email
